# Multi-Task Pulmonologist Specialist GPU Training
### MIMIC-IV Clinical + Notes + ED Unified Dataset (96,963 Encounters)

This notebook fine-tunes **PubMedBERT** (`microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract`) for multi-task clinical pulmonology inference:
1. **Pulmonary Disease Identification** (6 multi-label classes: `asthma`, `copd`, `pneumonia`, `respiratory_failure`, `pleural_effusion`, `pulmonary_embolism`)
2. **Chief Complaint Triage** (11 classes: `dyspnea`, `cough`, `wheezing`, `chest pain`, `fever`, `hemoptysis`, `sputum production`, `fatigue`, `altered mental status`, `syncope`, `respiratory assessment`)
3. **Clinical Urgency Stratification** (3 classes: `Emergency`, `Urgent`, `Routine`)

**Scientific Guarantees**: 0 patient overlap across splits (`subject_id` isolated), 0 target-label leakage, exact threshold optimization on validation set only, and INT8 ONNX export for edge deployment.

In [ ]:
# 1. Environment & GPU Setup
!nvidia-smi
!pip install -q transformers torch scikit-learn onnx onnxruntime matplotlib seaborn pandas

In [ ]:
# 2. Dataset Setup
import os, zipfile

os.makedirs('datasets/processed/pulmonologist', exist_ok=True)

if os.path.exists('pulmonologist_splits.zip'):
    print("Unzipping pulmonologist_splits.zip...")
    with zipfile.ZipFile('pulmonologist_splits.zip', 'r') as z:
        z.extractall('datasets/processed/pulmonologist')
elif os.path.exists('/content/drive/MyDrive/pulmonologist_splits.zip'):
    print("Unzipping from Google Drive...")
    with zipfile.ZipFile('/content/drive/MyDrive/pulmonologist_splits.zip', 'r') as z:
        z.extractall('datasets/processed/pulmonologist')
elif not os.path.exists('datasets/processed/pulmonologist/train_split.json'):
    print("Please upload pulmonologist_splits.zip or train_split.json, val_split.json, test_split.json")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as z:
                z.extractall('datasets/processed/pulmonologist')

print("Datasets ready in datasets/processed/pulmonologist:")
for f in os.listdir('datasets/processed/pulmonologist'):
    if f.endswith('.json'):
        sz = os.path.getsize(os.path.join('datasets/processed/pulmonologist', f)) / (1024**2)
        print(f"  -> {f} ({sz:.2f} MB)")

In [ ]:
# 3. Reproducibility & Device Initialization
import time, json, random, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score, average_precision_score, hamming_loss, accuracy_score
import matplotlib.pyplot as plt, seaborn as sns
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running training on device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# 4. Taxonomy & Multi-Task Dataset
PULM_DISEASE_CLASSES = ['asthma', 'copd', 'pneumonia', 'respiratory_failure', 'pleural_effusion', 'pulmonary_embolism']
PULM_COMPLAINT_CLASSES = [
    'dyspnea', 'cough', 'wheezing', 'chest pain', 'fever', 'hemoptysis',
    'sputum production', 'fatigue', 'altered mental status', 'syncope', 'respiratory assessment'
]
URGENCY_CLASSES = ['Emergency', 'Urgent', 'Routine']

DISEASE_MAP = {lbl: idx for idx, lbl in enumerate(PULM_DISEASE_CLASSES)}
COMPLAINT_MAP = {lbl: idx for idx, lbl in enumerate(PULM_COMPLAINT_CLASSES)}
URGENCY_MAP = {lbl: idx for idx, lbl in enumerate(URGENCY_CLASSES)}

class PulmonologistDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=384, desc="Dataset"):
        self.records = records
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.texts = [r.get('text', 'No clinical narrative.') for r in records]
        
        self.disease_targets = []
        for r in records:
            vec = np.zeros(len(PULM_DISEASE_CLASSES), dtype=np.float32)
            lbls = r.get('pulmonary_disease_labels', [])
            for d in lbls:
                if d in DISEASE_MAP:
                    vec[DISEASE_MAP[d]] = 1.0
            if vec.sum() == 0:
                vec[DISEASE_MAP['pneumonia']] = 1.0
            self.disease_targets.append(vec)
            
        self.complaint_targets = [COMPLAINT_MAP.get(r.get('chief_complaint', 'respiratory assessment'), 10) for r in records]
        self.urgency_targets = [URGENCY_MAP.get(r.get('urgency', 'Emergency'), 0) for r in records]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'disease_labels': torch.tensor(self.disease_targets[idx], dtype=torch.float),
            'complaint_label': torch.tensor(self.complaint_targets[idx], dtype=torch.long),
            'urgency_label': torch.tensor(self.urgency_targets[idx], dtype=torch.long)
        }

In [ ]:
# 5. Multi-Task PubMedBERT Neural Architecture
class PulmonologistSpecialistModel(nn.Module):
    def __init__(self, model_name='microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract', num_diseases=6, num_complaints=11, num_urgencies=3):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden_size = self.transformer.config.hidden_size
        
        self.fc_dis1 = nn.Linear(hidden_size, 256)
        self.fc_dis2 = nn.Linear(256, num_diseases)
        
        self.fc_comp1 = nn.Linear(hidden_size, 256)
        self.fc_comp2 = nn.Linear(256, num_complaints)
        
        self.fc_urg1 = nn.Linear(hidden_size, 256)
        self.fc_urg2 = nn.Linear(256, num_urgencies)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs[0][:, 0, :]
        
        d = self.dropout(self.relu(self.fc_dis1(cls_rep)))
        disease_logits = self.fc_dis2(d)
        
        c = self.dropout(self.relu(self.fc_comp1(cls_rep)))
        complaint_logits = self.fc_comp2(c)
        
        u = self.dropout(self.relu(self.fc_urg1(cls_rep)))
        urgency_logits = self.fc_urg2(u)
        
        return disease_logits, complaint_logits, urgency_logits

In [ ]:
# 6. Data Loading & Class Weight Configuration
DATA_DIR = 'datasets/processed/pulmonologist'
OUTPUT_DIR = 'experiments/pulmonologist_mimic_final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f'{DATA_DIR}/train_split.json', 'r', encoding='utf-8') as f: train_recs = json.load(f)
with open(f'{DATA_DIR}/val_split.json', 'r', encoding='utf-8') as f: val_recs = json.load(f)
with open(f'{DATA_DIR}/test_split.json', 'r', encoding='utf-8') as f: test_recs = json.load(f)
print(f"Cohort Scale: Train={len(train_recs)}, Val={len(val_recs)}, Test={len(test_recs)}")

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = PulmonologistDataset(train_recs, tokenizer, max_len=384, desc='Train Split')
val_dataset = PulmonologistDataset(val_recs, tokenizer, max_len=384, desc='Val Split')
test_dataset = PulmonologistDataset(test_recs, tokenizer, max_len=384, desc='Test Split')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, pin_memory=(DEVICE.type == 'cuda'))
val_loader = DataLoader(val_dataset, batch_size=32, pin_memory=(DEVICE.type == 'cuda'))
test_loader = DataLoader(test_dataset, batch_size=32, pin_memory=(DEVICE.type == 'cuda'))

# Class weights for severe positive/negative imbalance (train set only)
d_matrix = np.array(train_dataset.disease_targets)
pos_counts = d_matrix.sum(axis=0)
neg_counts = len(d_matrix) - pos_counts
pos_weights = np.sqrt(neg_counts / np.maximum(pos_counts, 1))
crit_disease = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float).to(DEVICE))

c_counts = np.bincount(train_dataset.complaint_targets, minlength=11)
c_weights = len(train_dataset) / (11.0 * np.maximum(c_counts, 1).astype(float))
crit_complaint = nn.CrossEntropyLoss(weight=torch.tensor(c_weights, dtype=torch.float).to(DEVICE))

u_counts = np.bincount(train_dataset.urgency_targets, minlength=3)
u_weights = len(train_dataset) / (3.0 * np.maximum(u_counts, 1).astype(float))
crit_urgency = nn.CrossEntropyLoss(weight=torch.tensor(u_weights, dtype=torch.float).to(DEVICE))

In [ ]:
# 7. Loss, Optimizer & Multi-Task Training Loop
EPOCHS = 3
LR = 2e-5
model = PulmonologistSpecialistModel(MODEL_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

def optimize_disease_thresholds(y_true, y_probs):
    thresholds = []
    for c in range(len(PULM_DISEASE_CLASSES)):
        best_f1, best_th = -1.0, 0.50
        for th in np.linspace(0.10, 0.90, 41):
            preds = (y_probs[:, c] >= th).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(y_true[:, c], preds, average='binary', zero_division=0)
            if f1 > best_f1: best_f1, best_th = f1, th
        thresholds.append(best_th)
    return np.array(thresholds)

best_val_f1 = -1.0
best_thresholds = None
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        ids, mask = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)
        d_t, c_t, u_t = batch['disease_labels'].to(DEVICE), batch['complaint_label'].to(DEVICE), batch['urgency_label'].to(DEVICE)
        
        optimizer.zero_grad()
        if DEVICE.type == 'cuda':
            with torch.amp.autocast('cuda'):
                d_l, c_l, u_l = model(ids, mask)
                loss = 1.0 * crit_disease(d_l, d_t) + 0.7 * crit_complaint(c_l, c_t) + 0.5 * crit_urgency(u_l, u_t)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            d_l, c_l, u_l = model(ids, mask)
            loss = 1.0 * crit_disease(d_l, d_t) + 0.7 * crit_complaint(c_l, c_t) + 0.5 * crit_urgency(u_l, u_t)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
        scheduler.step()
        total_loss += loss.item()
        if (step + 1) % 500 == 0 or step == len(train_loader) - 1:
            print(f"Epoch {epoch+1}/{EPOCHS} | Step {step+1}/{len(train_loader)} | Batch Loss: {loss.item():.4f}")
            
    # Validation Evaluation
    model.eval()
    val_d_trues, val_d_probs = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids, mask = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)
            d_l, _, _ = model(ids, mask)
            val_d_trues.append(batch['disease_labels'].numpy())
            val_d_probs.append(torch.sigmoid(d_l).cpu().numpy())
    val_d_trues = np.vstack(val_d_trues)
    val_d_probs = np.vstack(val_d_probs)
    opt_th = optimize_disease_thresholds(val_d_trues, val_d_probs)
    val_preds = (val_d_probs >= opt_th).astype(int)
    _, _, val_macro_f1, _ = precision_recall_fscore_support(val_d_trues, val_preds, average='macro', zero_division=0)
    print(f"=== Epoch {epoch+1} Completed | Val Disease Macro-F1: {val_macro_f1:.4f} ===")
    
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_thresholds = opt_th
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        print(f"--> Saved new best checkpoint to {OUTPUT_DIR}/best_model.pt")

print(f"\nTraining finished in {(time.time() - start_time)/60:.2f} minutes.")

In [ ]:
# 8. Comprehensive Test Split Evaluation
ckpt_path = f"{OUTPUT_DIR}/best_model.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()

test_d_trues, test_d_probs = [], []
test_c_trues, test_c_preds = [], []
test_u_trues, test_u_preds = [], []

with torch.no_grad():
    for batch in test_loader:
        ids, mask = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)
        d_l, c_l, u_l = model(ids, mask)
        test_d_trues.append(batch['disease_labels'].numpy())
        test_d_probs.append(torch.sigmoid(d_l).cpu().numpy())
        test_c_trues.extend(batch['complaint_label'].numpy().tolist())
        test_c_preds.extend(torch.argmax(c_l, dim=1).cpu().numpy().tolist())
        test_u_trues.extend(batch['urgency_label'].numpy().tolist())
        test_u_preds.extend(torch.argmax(u_l, dim=1).cpu().numpy().tolist())

test_d_trues = np.vstack(test_d_trues)
test_d_probs = np.vstack(test_d_probs)
test_d_preds = (test_d_probs >= best_thresholds).astype(int)

print("\n=======================================================")
print("HEAD 1: PULMONARY DISEASE CLASSIFICATION REPORT (6 CLASSES)")
print("=======================================================")
print(classification_report(test_d_trues, test_d_preds, target_names=PULM_DISEASE_CLASSES, zero_division=0))

print("\n=======================================================")
print("HEAD 2: CHIEF COMPLAINT CLASSIFICATION REPORT (11 CLASSES)")
print("=======================================================")
print(classification_report(test_c_trues, test_c_preds, target_names=PULM_COMPLAINT_CLASSES, zero_division=0))

print("\n=======================================================")
print("HEAD 3: CLINICAL URGENCY CLASSIFICATION REPORT (3 CLASSES)")
print("=======================================================")
print(classification_report(test_u_trues, test_u_preds, target_names=URGENCY_CLASSES, zero_division=0))

In [ ]:
# 9. ONNX Export & INT8 Dynamic Quantization
class PulmonologistSpecialistONNX(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.transformer = trained_model.transformer
        self.w_dis1 = nn.Parameter(trained_model.fc_dis1.weight.t().clone())
        self.b_dis1 = nn.Parameter(trained_model.fc_dis1.bias.clone())
        self.w_dis2 = nn.Parameter(trained_model.fc_dis2.weight.t().clone())
        self.b_dis2 = nn.Parameter(trained_model.fc_dis2.bias.clone())
        
        self.w_comp1 = nn.Parameter(trained_model.fc_comp1.weight.t().clone())
        self.b_comp1 = nn.Parameter(trained_model.fc_comp1.bias.clone())
        self.w_comp2 = nn.Parameter(trained_model.fc_comp2.weight.t().clone())
        self.b_comp2 = nn.Parameter(trained_model.fc_comp2.bias.clone())
        
        self.w_urg1 = nn.Parameter(trained_model.fc_urg1.weight.t().clone())
        self.b_urg1 = nn.Parameter(trained_model.fc_urg1.bias.clone())
        self.w_urg2 = nn.Parameter(trained_model.fc_urg2.weight.t().clone())
        self.b_urg2 = nn.Parameter(trained_model.fc_urg2.bias.clone())

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_rep = outputs[0][:, 0, :].reshape(-1, self.transformer.config.hidden_size)
        
        d = torch.matmul(cls_rep, self.w_dis1) + self.b_dis1
        logits_disease = torch.matmul(torch.relu(d), self.w_dis2) + self.b_dis2
        
        c = torch.matmul(cls_rep, self.w_comp1) + self.b_comp1
        logits_complaint = torch.matmul(torch.relu(c), self.w_comp2) + self.b_comp2
        
        u = torch.matmul(cls_rep, self.w_urg1) + self.b_urg1
        logits_urgency = torch.matmul(torch.relu(u), self.w_urg2) + self.b_urg2
        
        return logits_disease, logits_complaint, logits_urgency

onnx_model = PulmonologistSpecialistONNX(model).to(DEVICE)
onnx_model.eval()
dummy_ids = torch.ones(1, 384, dtype=torch.long).to(DEVICE)
dummy_mask = torch.ones(1, 384, dtype=torch.long).to(DEVICE)

onnx_fp32_path = "backend/models/pulmonologist_final.onnx"
onnx_int8_path = "backend/models/pulmonologist_final_quant.onnx"

torch.onnx.export(
    onnx_model,
    (dummy_ids, dummy_mask),
    onnx_fp32_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["disease_logits", "complaint_logits", "urgency_logits"],
    dynamic_axes={"input_ids": {0: "batch_size", 1: "sequence_length"}, "attention_mask": {0: "batch_size", 1: "sequence_length"}},
    opset_version=18
)
print(f"Exported FP32 ONNX to {onnx_fp32_path}")

quantize_dynamic(model_input=onnx_fp32_path, model_output=onnx_int8_path, weight_type=QuantType.QUInt8)
print(f"Exported INT8 Quantized ONNX to {onnx_int8_path}")